<a href="https://colab.research.google.com/github/damanadidier/DI_Bootcamp/blob/main/Exercises_XP_Day4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: LoRA Implementation Lab
Replace each `TODO` before running the next section.

## What you'll learn

- The fundamentals of LoRA (Low-Rank Adaptation) and why it helps churn out efficient fine-tunes.
- How to implement LoRA matrices `A` and `B`, plus how to wrap existing `nn.Linear` layers.
- Differences between standard linear layers, LoRA-enhanced layers, and merged-weight alternatives.
- How to freeze base parameters so that only the LoRA adapters receive updates.

## What you will create

- A reusable `LoRALayer` module and two linear wrappers (`LinearWithLoRA`, `LinearWithLoRAMerged`).
- A 3-layer MLP that can be swapped between standard and LoRA-enhanced variants.
- A minimal MNIST training loop plus accuracy helpers to compare frozen vs. fully-trainable adapters.
- A workflow to freeze baseline weights and fine-tune only the LoRA layers.

> **Learning point**  
> Keep the student and teacher notebooks open side by side. Follow the numbered exercises, run setup only once, and watch tensor shapes as you add LoRA adapters.

# Part 0: Environment Setup

Install the CPU-friendly PyTorch stack plus torchvision for MNIST. Reuse caches across reruns to save time.

In [11]:
%pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BASE_SEED = 123
torch.manual_seed(BASE_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


# Exercise 1: Implement `LoRALayer`

Create the low-rank matrices `A` and `B`, scale them with `alpha`, and test the module on a toy tensor.

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        # Standard deviation for low-rank initialization
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        # A is initialized with Gaussian noise
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        # B is initialized to zero to ensure delta W is zero at start
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x: (batch, in_dim) @ A: (in_dim, rank) @ B: (rank, out_dim)
        x = (x @ self.A @ self.B) * self.alpha
        return x

# Hyperparameters for the sandbox test
random_seed = 123
in_dim = 10
out_dim = 5
rank = 2
alpha = 1

torch.manual_seed(random_seed)
layer = LoRALayer(in_dim, out_dim, rank, alpha)
x = torch.randn(1, in_dim)

print(f"Input shape: {x.shape}")
print(layer)
print("Original output (should be zero because B=0):", layer(x))

Input shape: torch.Size([1, 10])
LoRALayer()
Original output (should be zero because B=0): tensor([[0., 0., 0., 0., 0.]], grad_fn=<MulBackward0>)


# Exercise 2: Wrap `nn.Linear` with LoRA

Combine a frozen linear projection plus a trainable `LoRALayer`. Confirm the adapter outputs add on top of the base logits.

In [13]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Sum the outputs of the frozen linear layer and the trainable LoRA path
        return self.linear(x) + self.lora(x)

base_linear = nn.Linear(in_dim, out_dim)
layer_lora_1 = LinearWithLoRA(base_linear, rank=rank, alpha=alpha)
print("LinearWithLoRA output:", layer_lora_1(x))

LinearWithLoRA output: tensor([[-0.3830,  0.7344,  0.2912, -0.3389,  0.0132]], grad_fn=<AddBackward0>)


# Exercise 3: Swap a simple network layer with LoRA

Start from a single-layer perceptron, then replace its linear block with `LinearWithLoRA`. The outputs should match before training because the LoRA adapters start at zero.

In [22]:
import torch
import torch.nn as nn

class SingleLayerNet(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.layer = nn.Linear(num_features, num_classes)

    def forward(self, x):
        return self.layer(x)

single_net = SingleLayerNet(num_features=10, num_classes=5)
sample_input = torch.randn(1, 10)

with torch.no_grad():
    baseline_output = single_net(sample_input)

# Swap the standard linear layer for our LoRA wrapper
single_net.layer = LinearWithLoRA(single_net.layer, rank=2, alpha=1)

with torch.no_grad():
    lora_output = single_net(sample_input)

# Check if outputs match (they should because B is initialized to zero)
match = torch.allclose(baseline_output, lora_output)
print("Outputs match before training?", match)

Outputs match before training? True


# Exercise 4: Merged-weight LoRA layer

Fuse the LoRA matrices with the frozen weights to create a drop-in linear layer that behaves exactly like `LinearWithLoRA`.

In [14]:
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha,
        )

    def forward(self, x):
        # Weight merging: W_new = W_old + alpha * (A @ B).T
        # We transpose because nn.Linear weights are (out_features, in_features)
        lora_weights = (self.lora.A @ self.lora.B).T * self.lora.alpha
        combined_weight = self.linear.weight + lora_weights
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(nn.Linear(in_dim, out_dim), rank=rank, alpha=alpha)
print("Merged LoRA output:", layer_lora_2(x))

Merged LoRA output: tensor([[-0.3014,  0.4505, -0.1323, -0.0352, -0.8533]],
       grad_fn=<AddmmBackward0>)


# Exercise 5: Build an MLP and prepare MNIST

Stack three linear layers with ReLU activations, then set up the MNIST loaders plus optimizer/state for pretraining.

In [16]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x):
        # Flatten images: (batch, 1, 28, 28) -> (batch, 784)
        x = x.view(-1, 28*28)
        return self.layers(x)

In [25]:
# Ensure environment variables are active
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Architecture configuration
num_features = 784 # 28x28 images
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

# Training settings
learning_rate = 0.005
num_epochs = 2

# Instantiate the model architecture
model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
)

model.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Device: {DEVICE}")
print(model)
print(f"Optimizer: {optimizer_pretrained}")

Device: cuda
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.005
    maximize: False
    weight_decay: 0
)


## Loading dataset

In [17]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Verification
for images, labels in train_loader:
    print('Batch shape:', images.shape)
    break

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.89MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.66MB/s]

Batch shape: torch.Size([64, 1, 28, 28])


## Define evaluation

In [8]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum().item()
    return (correct_pred / num_examples) * 100

## Training

In [29]:
import time

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print('Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [31]:
import time

# Re-defining optimizer to ensure it exists in the current session
optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Pre-train the original base model
print("Starting baseline training...")
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f"Test accuracy (Original Model): {compute_accuracy(model, test_loader, DEVICE):.2f}%")

Starting baseline training...


/tmp/ipykernel_762/510695364.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/002 | Batch 000/938 | Loss: 2.3119
Epoch: 001/002 | Batch 400/938 | Loss: 0.1790
Epoch: 001/002 | Batch 800/938 | Loss: 0.1092
Epoch: 001/002 training accuracy: 97.02%
Time elapsed: 0.24 min
Epoch: 002/002 | Batch 000/938 | Loss: 0.1182
Epoch: 002/002 | Batch 400/938 | Loss: 0.0524
Epoch: 002/002 | Batch 800/938 | Loss: 0.1300
Epoch: 002/002 training accuracy: 97.40%
Time elapsed: 0.47 min
Total Training Time: 0.47 min
Test accuracy (Original Model): 96.53%


# Replacing Linear with LoRA Layers

In [30]:
import copy

# Ensure the model exists before copying
if 'model' not in globals():
    model = MultilayerPerceptron(784, 128, 64, 10).to(DEVICE)

# Create a copy for LoRA adaptation
model_lora = copy.deepcopy(model)

# Replace layers with LoRA variants
model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)

model_lora.to(DEVICE)

print("LoRA Model Architecture:")
print(model_lora)

LoRA Model Architecture:
MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=64, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=64, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)


## Freezing the Original Linear Layers

In [32]:
def freeze_linear_layers(model):
    for name, module in model.named_modules():
        # If it is a Merged LoRA layer, we only want to freeze the internal 'linear' part
        if isinstance(module, LinearWithLoRAMerged):
            for param in module.linear.parameters():
                param.requires_grad = False
        # For standard linear layers that aren't wrapped
        elif isinstance(module, nn.Linear):
            for param in module.parameters():
                param.requires_grad = False

# Freeze the base weights of the LoRA-augmented model
freeze_linear_layers(model_lora)

# Verify gradients: only A and B parameters within LoRALayer should be True
for name, param in model_lora.named_parameters():
    print(f"{name:40} | Trainable: {param.requires_grad}")

layers.0.linear.weight                   | Trainable: False
layers.0.linear.bias                     | Trainable: False
layers.0.lora.A                          | Trainable: True
layers.0.lora.B                          | Trainable: True
layers.2.linear.weight                   | Trainable: False
layers.2.linear.bias                     | Trainable: False
layers.2.lora.A                          | Trainable: True
layers.2.lora.B                          | Trainable: True
layers.4.linear.weight                   | Trainable: False
layers.4.linear.bias                     | Trainable: False
layers.4.lora.A                          | Trainable: True
layers.4.lora.B                          | Trainable: True


In [33]:
# Re-verify and fine-tune the LoRA-enhanced model
if 'model_lora' in globals():
    # We filter the optimizer to only update parameters where requires_grad=True (the LoRA A/B matrices)
    optimizer_lora = torch.optim.Adam(filter(lambda p: p.requires_grad, model_lora.parameters()), lr=learning_rate)

    print("Starting LoRA Fine-tuning...")
    train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)

    print(f"Final Test accuracy (LoRA fine-tuned): {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")
    print(f"Reference Test accuracy (Original): {compute_accuracy(model, test_loader, DEVICE):.2f}%")
else:
    print("Error: model_lora not found. Please re-run cell 46a08524.")

Starting LoRA Fine-tuning...
Epoch: 001/002 | Batch 000/938 | Loss: 2.3016
Epoch: 001/002 | Batch 400/938 | Loss: 1.1273
Epoch: 001/002 | Batch 800/938 | Loss: 0.6625
Epoch: 001/002 training accuracy: 78.73%
Time elapsed: 0.25 min
Epoch: 002/002 | Batch 000/938 | Loss: 0.3292
Epoch: 002/002 | Batch 400/938 | Loss: 0.3630
Epoch: 002/002 | Batch 800/938 | Loss: 0.7108
Epoch: 002/002 training accuracy: 85.42%
Time elapsed: 0.50 min
Total Training Time: 0.50 min
Final Test accuracy (LoRA fine-tuned): 85.57%
Reference Test accuracy (Original): 96.53%
